In [1]:
import pandas as pd
import numpy as np
import time

In [2]:
df = pd.read_csv(r"C:\Users\rajat\OneDrive\Desktop\netflix_titles.csv")

In [3]:
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   show_id       8807 non-null   str  
 1   type          8807 non-null   str  
 2   title         8807 non-null   str  
 3   director      6173 non-null   str  
 4   cast          7982 non-null   str  
 5   country       7976 non-null   str  
 6   date_added    8797 non-null   str  
 7   release_year  8807 non-null   int64
 8   rating        8803 non-null   str  
 9   duration      8804 non-null   str  
 10  listed_in     8807 non-null   str  
 11  description   8807 non-null   str  
dtypes: int64(1), str(11)
memory usage: 825.8 KB


In [5]:
netflix_titles_copy = df.copy()

In [6]:
netflix_titles_copy['director'] = netflix_titles_copy['director'].fillna('Unknown')
netflix_titles_copy['cast'] = netflix_titles_copy['cast'].fillna('Unknown')
netflix_titles_copy['country'] = netflix_titles_copy['country'].fillna('Unknown')
netflix_titles_copy['rating'] = netflix_titles_copy['rating'].fillna('Not Rated')
netflix_titles_copy['duration'] = netflix_titles_copy['duration'].fillna('Unknown')
netflix_titles_copy['date_added'] = netflix_titles_copy['date_added'].fillna('Not Available')

In [7]:
cols = ['title', 'director', 'cast', 'listed_in', 'description']

for col in cols:
    netflix_titles_copy[col] = netflix_titles_copy[col].str.lower()

In [8]:
netflix_titles_copy[['title', 'director']].head()

,title,director
0,dick johnson is dead,kirsten johnson
1,blood & water,unknown
2,ganglands,julien leclercq
3,jailbirds new orleans,unknown
4,kota factory,unknown


In [9]:
netflix_titles_copy['combined_features'] = (
    netflix_titles_copy['listed_in'] + ' ' +
    netflix_titles_copy['description'] + ' ' +
    netflix_titles_copy['cast'] + ' ' +
    netflix_titles_copy['director']
)

In [10]:
netflix_titles_copy[['title', 'combined_features']].head()

,title,combined_features
0,dick johnson is dead,documentaries as her father nears the end of h...
1,blood & water,"international tv shows, tv dramas, tv mysterie..."
2,ganglands,"crime tv shows, international tv shows, tv act..."
3,jailbirds new orleans,"docuseries, reality tv feuds, flirtations and ..."
4,kota factory,"international tv shows, romantic tv shows, tv ..."


In [11]:
netflix_titles_copy['combined_features'] = netflix_titles_copy['combined_features'].str.replace(',', '')

In [12]:
netflix_titles_copy['combined_features'].head()

0    documentaries as her father nears the end of h...
1    international tv shows tv dramas tv mysteries ...
2    crime tv shows international tv shows tv actio...
3    docuseries reality tv feuds flirtations and to...
4    international tv shows romantic tv shows tv co...
Name: combined_features, dtype: str

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [14]:
tfidf = TfidfVectorizer(stop_words='english')

tfidf_matrix = tfidf.fit_transform(netflix_titles_copy['combined_features'])

In [15]:
tfidf_matrix.shape

(8807, 49955)

In [16]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [17]:
cosine_sim.shape

(8807, 8807)

In [18]:
indices = pd.Series(netflix_titles_copy.index, index=netflix_titles_copy['title']).drop_duplicates()

In [19]:
indices['kota factory']

np.int64(4)

In [20]:
def recommend(title):
    title = title.lower()
    
    if title not in indices:
        return "Movie not found!"
    
    idx = indices[title]
    
    sim_scores = list(enumerate(cosine_sim[idx]))
    
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    sim_scores = sim_scores[1:6]
    
    movie_indices = [i[0] for i in sim_scores]
    scores = [i[1] for i in sim_scores]
    
    result = netflix_titles_copy[['title']].iloc[movie_indices].copy()
    result['similarity_score'] = scores
    
    return result

In [21]:
recommend("kota factory")

,title,similarity_score
8775,yeh meri family,0.156118
3466,girls hostel,0.146440
2353,chaman bahaar,0.140001
2472,betaal,0.124861
266,the creative indians,0.117282


In [22]:
!pip install requests

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import requests

In [3]:
API_KEY = "4f1af0428c8004ab7c62e6bd5be74ccb"

In [18]:
##netflix_titles_copy = pd.read_csv(r"C:\Users\rajat\OneDrive\Desktop\netflix_titles.csv")
netflix_titles_copy = pd.read_csv("netflix_with_posters.csv")

In [20]:
def fetch_poster(movie_name):
    try:
        movie_url = f"https://api.themoviedb.org/3/search/movie?api_key={API_KEY}&query={movie_name}"
        movie_data = requests.get(movie_url, timeout=5).json()
        
        if movie_data.get('results'):
            poster_path = movie_data['results'][0].get('poster_path')
            if poster_path:
                return "https://image.tmdb.org/t/p/w500" + poster_path
        
        tv_url = f"https://api.themoviedb.org/3/search/tv?api_key={API_KEY}&query={movie_name}"
        tv_data = requests.get(tv_url, timeout=5).json()
        
        if tv_data.get('results'):
            poster_path = tv_data['results'][0].get('poster_path')
            if poster_path:
                return "https://image.tmdb.org/t/p/w500" + poster_path
    
    except:
        return "Error"
    
    return "No Image Found"

In [21]:
netflix_titles_copy['poster_url'].notnull().sum()

np.int64(8807)

In [22]:
batch_size = 50

start_index = netflix_titles_copy['poster_url'].notnull().sum()
print(f"Resuming from row {start_index}")

for i in range(start_index, len(netflix_titles_copy)):
    
    title = netflix_titles_copy.loc[i, 'title']
    
    poster = fetch_poster(title)
    netflix_titles_copy.loc[i, 'poster_url'] = poster
    
    print(f"Row {i} done")  
    
    if i % batch_size == 0:
        netflix_titles_copy.to_csv("netflix_with_posters.csv", index=False)
        print(f"Saved up to row {i}")

Resuming from row 8807


In [23]:
netflix_titles_copy['poster_url'].notnull().sum()

np.int64(8807)

In [24]:
netflix_titles_copy.to_csv("netflix_with_posters.csv", index=False)
print("Final save complete")

Final save complete
